# Homework 4 — Evaluation

LLM Zoomcamp 2026 · Cohort

HW2 built three ways to search the course lessons: keyword, vector, and hybrid. This homework measures which one actually works, using Hit Rate and MRR against a ground-truth dataset instead of eyeballing a few examples.

**Knowledge base:** same 72 lesson markdown pages from commit `8c1834d`, chunked exactly as in HW2 (`size=2000, step=1000` → 295 chunks).

## Setup

We reuse HW2's helper files directly (`embedder.py`, `download.py`, the downloaded ONNX model) via `sys.path`, instead of copying the ~90MB model directory again.

New dependencies for this module (LLM calls with structured output, CSV handling):

```bash
uv add openai pydantic python-dotenv pandas
```

In [1]:
import sys
import json

import numpy as np
import pandas as pd
import minsearch
from tqdm.auto import tqdm
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

from gitsource import GithubRepositoryDataReader, chunk_documents

# embedder.py and the downloaded ONNX model live in hw2-vector-search/
sys.path.insert(0, "../hw2-vector-search")
from embedder import Embedder

# llm_structured / calc_price live in evaluation_utils.py, downloaded from the course repo
from evaluation_utils import llm_structured

load_dotenv()
openai_client = OpenAI()

/Users/carlosibarra/projects/llm-zoomcamp-2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading and chunking the data

Identical to HW2: same 72 lesson pages, same `size=2000, step=1000` chunking → 295 chunks.

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} lesson pages")

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Loaded 72 lesson pages
Total chunks: 295


## Rebuilding search from HW2

We rebuild the keyword index (`minsearch.Index`) and the vector index (`minsearch.VectorSearch`) over the chunks, then wrap each one in a function with the same `(query, num_results=5)` signature. That way the evaluation code below can call any of the three search methods the same way.

In [3]:
embedder = Embedder(path="../hw2-vector-search/models/Xenova/all-MiniLM-L6-v2")

print("Encoding all chunks (this takes a moment)...")
X = np.array(embedder.encode_batch([c["content"] for c in chunks]))
print(f"Embedding matrix shape: {X.shape}")

ki = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
ki.fit(chunks)

vs = minsearch.VectorSearch(keyword_fields=["filename"])
vs.fit(X, chunks)


def text_search(query, num_results=5):
    return ki.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vs.search(v, num_results=num_results)

Encoding all chunks (this takes a moment)...


Embedding matrix shape: (295, 384)


For hybrid search, we reuse the RRF (Reciprocal Rank Fusion) function from HW2: it combines the keyword and vector result lists by rank position (not raw score), so a document that ranks well in *both* lists wins even if it's not #1 in either.

In [4]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q1. Generating ground truth questions

To evaluate search, we need a dataset of questions paired with the document that should answer each one. We generate these with an LLM using structured output: instead of asking for a paragraph and parsing it out, we ask the model to return a Python object matching a schema we define with Pydantic. That gives us the list of questions directly, no parsing needed.

We ask for 5 questions per lesson page, phrased the way a student would actually type them into a search box, not copied from the page's own wording. That keeps the evaluation realistic.

Generating this for all 72 pages costs money and takes time, so for Q1 we only run it on 3 pages, just to check the token cost per call.

In [5]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

q1_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

input_tokens_list = []

for fname in q1_filenames:
    doc = next(d for d in documents if d["filename"] == fname)
    user_prompt = json.dumps(doc)

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    print(f"{fname}")
    print(f"  input_tokens: {usage.input_tokens}")
    print(f"  sample question: {result.questions[0]}")

    input_tokens_list.append(usage.input_tokens)

avg_input_tokens = sum(input_tokens_list) / len(input_tokens_list)
print(f"\nAverage input tokens across 3 calls: {avg_input_tokens:.1f}")

01-agentic-rag/lessons/01-intro.md
  input_tokens: 1020
  sample question: What problem does retrieval-augmented generation solve for language models?


01-agentic-rag/lessons/02-environment.md
  input_tokens: 1286
  sample question: What do I need installed before starting this module besides Jupyter, and what Python version should I use?


01-agentic-rag/lessons/03-rag.md
  input_tokens: 1753
  sample question: Why doesn’t a plain LLM answer course-specific questions well, even if it can answer general ones?

Average input tokens across 3 calls: 1353.0


**Answer Q1: `1400`**

The average came out to about 1350 input tokens per call. Each request sends the full page content as JSON plus the instructions, so it's easily over a thousand tokens - well into the 1400 range, not close to 140 or 14000.

## The full ground truth

Generating questions for all 72 pages ourselves is out of scope here. The course already ran this same process and shared the result: 360 questions covering every page. We just download it.

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
```

In [6]:
df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
print(f"Ground truth records: {len(ground_truth)}")
ground_truth[0]

Ground truth records: 360


{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Q2/Q3. First result: text search vs vector search

Take the first ground truth question and compare which page each search method puts on top.

In [7]:
q = ground_truth[0]["question"]
correct_filename = ground_truth[0]["filename"]
print(f"Question: {q}")
print(f"Correct filename: {correct_filename}")

text_top1 = text_search(q)[0]["filename"]
vector_top1 = vector_search(q)[0]["filename"]

print(f"\ntext_search   top-1: {text_top1}")
print(f"vector_search top-1: {vector_top1}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Correct filename: 01-agentic-rag/lessons/01-intro.md

text_search   top-1: 01-agentic-rag/lessons/03-rag.md
vector_search top-1: 01-agentic-rag/lessons/01-intro.md


**Answer Q2: `01-agentic-rag/lessons/03-rag.md`**

**Answer Q3: `01-agentic-rag/lessons/01-intro.md`**

The question came from `01-intro.md`. Vector search puts it first, keyword search doesn't. This is just one query though, so it doesn't tell us much about which method is better overall - that's why we run the full evaluation next.

## Evaluation metrics

We turn search results into two numbers, computed across the whole ground truth dataset:

- **Hit Rate**: fraction of questions where the correct page shows up anywhere in the top results.
- **MRR (Mean Reciprocal Rank)**: average of `1 / rank` for the first correct result (0 if it's not found at all). Unlike Hit Rate, this also cares about position, so finding the right page at #1 scores higher than finding it at #5. MRR is always lower than or equal to Hit Rate.

Our relevance check compares `filename` instead of `id`, since the course's version works on FAQ records and ours works on lesson-page chunks.

In [8]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])
    return [int(d["filename"] == filename) for d in results]


def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]


def hit_rate(relevance_total):
    return sum(1 for line in relevance_total if 1 in line) / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4. Evaluating text search

In [9]:
text_metrics = evaluate(ground_truth, text_search)
text_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

 33%|███▎      | 118/360 [00:00<00:00, 1124.73it/s]

 67%|██████▋   | 240/360 [00:00<00:00, 1176.91it/s]

 99%|█████████▉| 358/360 [00:00<00:00, 1056.11it/s]

100%|██████████| 360/360 [00:00<00:00, 1077.21it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

**Answer Q4: `0.76`**

Hit Rate came out to 0.758: keyword search finds the correct page somewhere in the top 5 for about 3 out of 4 questions.

## Q5. Evaluating vector search

The lesson only evaluated keyword search. Vector search was left for the homework.

In [10]:
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

  7%|▋         | 26/360 [00:00<00:01, 249.36it/s]

 15%|█▍        | 53/360 [00:00<00:01, 258.79it/s]

 22%|██▎       | 81/360 [00:00<00:01, 265.14it/s]

 30%|███       | 109/360 [00:00<00:00, 268.38it/s]

...

 83%|████████▎ | 300/360 [00:01<00:00, 192.27it/s]

 89%|████████▉ | 321/360 [00:01<00:00, 190.68it/s]

 95%|█████████▌| 342/360 [00:01<00:00, 183.82it/s]

100%|██████████| 360/360 [00:01<00:00, 226.60it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

**Answer Q5: `0.55`**

MRR came out to 0.549. Vector search actually has a lower Hit Rate than text search on this dataset (0.725 vs 0.758), even though it can catch semantically related pages that don't share any words with the question, like we saw in Q2/Q3. Keyword matching still wins more often on raw recall here.

## Q6. Tuning hybrid search

The RRF constant `k` controls how much the top ranks matter. A small `k` makes the gap between position #1 and #2 bigger, so ranking first counts for a lot more. We sweep `k` over the full ground truth instead of assuming the RRF paper's default of 60 is right for this data.

In [11]:
k_results = []

for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k={k}: {result}")
    k_results.append({"k": k, **result})

df_k_results = pd.DataFrame(k_results)
df_k_results

  0%|          | 0/360 [00:00<?, ?it/s]

  5%|▍         | 17/360 [00:00<00:02, 169.57it/s]

 11%|█         | 39/360 [00:00<00:01, 198.95it/s]

 16%|█▋        | 59/360 [00:00<00:02, 126.87it/s]

 22%|██▏       | 79/360 [00:00<00:01, 146.83it/s]

...

 92%|█████████▏| 332/360 [00:01<00:00, 189.34it/s]

 98%|█████████▊| 352/360 [00:01<00:00, 176.19it/s]

100%|██████████| 360/360 [00:01<00:00, 188.59it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


,k,hit_rate,mrr
0,1,0.838889,0.648194
1,50,0.836111,0.637917
2,100,0.836111,0.637917
3,200,0.836111,0.637917


In [12]:
# Best MRR; on a tie, prefer the smallest k
best_row = df_k_results.sort_values(["mrr", "k"], ascending=[False, True]).iloc[0]
print(f"Best k: {int(best_row['k'])} (mrr={best_row['mrr']:.4f}, hit_rate={best_row['hit_rate']:.4f})")

Best k: 1 (mrr=0.6482, hit_rate=0.8389)


**Answer Q6: `k=1`**

`k=1` gives the best MRR (0.648), ahead of k=50/100/200, which all tie at 0.638. On this dataset, the top result from each individual search method is usually trustworthy, so rewarding it more heavily pays off. The paper's default of 60 isn't automatically the right choice here.

## Summary of answers

| Question | Answer |
|---|---|
| Q1. Average input tokens (3 pages) | `1400` |
| Q2. `text_search` top-1 filename for `ground_truth[0]` | `01-agentic-rag/lessons/03-rag.md` |
| Q3. `vector_search` top-1 filename for `ground_truth[0]` | `01-agentic-rag/lessons/01-intro.md` |
| Q4. Hit Rate of `text_search` | `0.76` |
| Q5. MRR of `vector_search` | `0.55` |
| Q6. Best `k` for `hybrid_search` (RRF) | `1` |

Submit at: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw4